# 04 - Audio Pipeline: Baseline Feature-Based Experiments

**Stage:** Audio-based fake news detection — exploration phase

**What this notebook does:**
- Extracts classical acoustic features to classify `voice_type` (real human voice vs TTS-generated)
- Exploratory analysis: boxplots, pairplots, correlation heatmap, PCA
- Trains baseline models: Logistic Regression, XGBoost, and an MLP

**Input:** extracted acoustic feature table
**Output:** baseline metrics used to justify moving to a deep model (Wav2Vec2) in `05_audio_wav2vec2_finetuning.ipynb`

> Part of the WAVE project — AI section (audio-based fake news detection).

In [ ]:
# --- Environment configuration ---
# Set BASE_DIR to wherever your data/model checkpoints live locally.
# If running on Google Colab, uncomment the two lines below.
# from google.colab import drive
# drive.mount('/content/drive')

import os
BASE_DIR = os.environ.get("WAVE_DATA_DIR", "./data")


# **تصنيف نوع الصوت نفسه → voice_type**

# هدفي بناء نموذج قادر على تحديد هل الملف الصوتي صوت بشري حقيقي أم مولد عبر TTS

# **استخراج الميزات الصوتية**

In [ ]:
import os
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm


base_dir = "BASE_DIR/Audio"


real_audio_dirs = [
    "Real audio/cnn/cnn",
    "Real audio/AlarabyTvSy/AlarabyTvSy",
    "Real audio/TelevisionSyria/TelevisionSyria",
    "Real audio/mixed_bilingual_files" 
]


tts_audio_dirs = [
    "generated data /VitsTTS",
    "generated data /piper_tts",
    "generated data /tts_outputs_v2"
]


def extract_features(file_path):
    try:
        y, sr = librosa.load(file_path, sr=None)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        zcr = librosa.feature.zero_crossing_rate(y)
        centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
        rms = librosa.feature.rms(y=y)

        return {
            "mfcc_mean": np.mean(mfcc),
            "mfcc_std": np.std(mfcc),
            "zcr_mean": np.mean(zcr),
            "centroid_mean": np.mean(centroid),
            "rms_mean": np.mean(rms)
        }
    except Exception as e:
        print(f"❌ Error with {file_path}: {e}")
        return None


data = []


for folder in real_audio_dirs:
    folder_path = os.path.join(base_dir, folder)
    if not os.path.exists(folder_path):
        continue

    for filename in tqdm(os.listdir(folder_path), desc=f"Processing REAL: {folder}"):
        if filename.endswith(".wav") or filename.endswith(".mp3"):
            path = os.path.join(folder_path, filename)
            feats = extract_features(path)
            if feats:
                feats['audio_file'] = filename
                feats['voice_type'] = 'real'
                data.append(feats)


for tts_folder in tts_audio_dirs:
    for subfolder in ['real', 'fake']:
        folder_path = os.path.join(base_dir, tts_folder, subfolder)
        if not os.path.exists(folder_path):
            print(f" Skip: {folder_path} doesn't exist")
            continue

        for filename in tqdm(os.listdir(folder_path), desc=f"Processing FAKE VOICE from {tts_folder}/{subfolder}"):
            if filename.endswith(".wav") or filename.endswith(".mp3"):
                path = os.path.join(folder_path, filename)
                feats = extract_features(path)
                if feats:
                    feats['audio_file'] = filename
                    feats['voice_type'] = 'fake'
                    data.append(feats)

df_audio_features = pd.DataFrame(data)

In [ ]:
df_audio_features

In [ ]:
df_audio_features.to_csv("BASE_DIR/Audio/audio_voice_type_features.csv", index=False)

# **Exploratory Analysis**

# Boxplot لكل ميزة حسب voice_type

# لحتى تشوف الفرق بالتوزيع بين real و fake

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

features_to_plot = ["mfcc_mean", "mfcc_std", "zcr_mean", "centroid_mean", "rms_mean"]

for feature in features_to_plot:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x="voice_type", y=feature, data=df_audio_features)
    plt.title(f"Distribution of {feature} by Voice Type")
    plt.grid(True)
    plt.show()


# **Pairplot (Scatter Matrix)**
# بتشوف فيه كل الميزات مقابل بعض، وبتلاحظ إذا فيه "فصل طبيعي" بين real و fake

In [ ]:
sns.pairplot(df_audio_features, hue="voice_type", vars=features_to_plot, plot_kws={'alpha': 0.6})
plt.suptitle("Pairwise Feature Relationships", y=1.02)
plt.show()


# **Correlation Heatmap**
# لتشوف إذا بعض الميزات مرتبطة ببعض، أو فيها تكرار

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df_audio_features[features_to_plot].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation between Audio Features")
plt.show()


# **PCA Plot لتقليل الأبعاد وتمثيل البيانات بنقطتين**

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(df_audio_features[features_to_plot])
df_pca = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
df_pca["voice_type"] = df_audio_features["voice_type"]

sns.scatterplot(data=df_pca, x="PC1", y="PC2", hue="voice_type")
plt.title("PCA of Audio Features")
plt.grid(True)
plt.show()


# **ML Section**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns


csv_path = "BASE_DIR/Audio/audio_voice_type_features.csv"
df = pd.read_csv(csv_path)

In [ ]:
df

In [ ]:

le = LabelEncoder()
df["voice_type_encoded"] = le.fit_transform(df["voice_type"])  # fake=0, real=1

In [ ]:
df

In [ ]:

X = df[["mfcc_mean", "mfcc_std", "zcr_mean", "centroid_mean", "rms_mean"]]
y = df["voice_type_encoded"]

In [ ]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:

model = LogisticRegression()
model.fit(X_train, y_train)


y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)


print(f"Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC Score: {auc:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["fake", "real"]))


plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues", xticklabels=["fake", "real"], yticklabels=["fake", "real"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Logistic Regression")
plt.show()


fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"AUC = {auc:.2f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.legend()
plt.grid(True)
plt.show()

# **XGBoost**

In [ ]:
import xgboost as xgb


dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)


params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'eta': 0.1,
    'max_depth': 5,
    'seed': 42
}


xgb_model = xgb.train(params, dtrain, num_boost_round=100)


y_proba = xgb_model.predict(dtest)
y_pred = (y_proba > 0.5).astype(int)


acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print("🔹 XGBoost نتائج:")
print(f"Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC Score: {auc:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["fake", "real"]))

 
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="YlGn", xticklabels=["fake", "real"], yticklabels=["fake", "real"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - XGBoost")
plt.show()


fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"AUC = {auc:.2f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - XGBoost")
plt.legend()
plt.grid(True)
plt.show()


# **NN -> MLP**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score


X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)


train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)


In [ ]:

class VoiceClassifier(nn.Module):
    def __init__(self):
        super(VoiceClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(5, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

model = VoiceClassifier()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


train_losses = []
val_losses = []

for epoch in range(30):
    model.train()
    running_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_losses.append(running_loss / len(train_loader))

   
    model.eval()
    with torch.no_grad():
        val_loss = 0
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
        val_losses.append(val_loss / len(test_loader))

    print(f"Epoch {epoch+1}: Train Loss = {train_losses[-1]:.4f}, Val Loss = {val_losses[-1]:.4f}")



In [ ]:

model.eval()
with torch.no_grad():
    y_pred_prob = model(X_test_tensor).numpy().flatten()
    y_pred = (y_pred_prob > 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_prob)

print("\n🔹 Neural Network النتائج:")
print(f"Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC Score: {auc:.4f}")


plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges", xticklabels=["fake", "real"], yticklabels=["fake", "real"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Neural Network")
plt.show()


fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"AUC = {auc:.2f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Neural Network")
plt.legend()
plt.grid(True)
plt.show()
